

*   Name : Molemi Molokoane
*   Email : iimillicent02@gmail.com



In [5]:
from google.colab import drive
drive.mount('/content/drive',  force_remount=True)

Mounted at /content/drive


In [6]:
import os
import sqlite3
import pandas as pd

# Look for the Olist CSVs in Colab (Drive / uploaded files) or locally
DATA_DIR = '/content/drive/MyDrive/Data-Science-Scholarship/archive'
print("Loading data from:", DATA_DIR)

conn = sqlite3.connect(":memory:")

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
}

for table, file in files.items():
    df = pd.read_csv(os.path.join(DATA_DIR, file))
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"{table}: {len(df):,} rows loaded")

Loading data from: /content/drive/MyDrive/Data-Science-Scholarship/archive
orders: 99,441 rows loaded
order_items: 112,650 rows loaded
products: 32,951 rows loaded
reviews: 99,224 rows loaded


In [7]:
def run(query):
    """Run a SQL query against the SQLite database and return a DataFrame."""
    return pd.read_sql_query(query, conn)

## 1. Distinct Statuses
Find the distinct list of status updates in the `orders` table.



In [8]:
query = """
SELECT DISTINCT order_status
FROM orders
ORDER BY order_status
"""

run(query)

,order_status
0,approved
1,canceled
2,created
3,delivered
4,invoiced
5,processing
6,shipped
7,unavailable


## 2. Price Summary
Find the minimum, average, and maximum price of items sold in the `order_items` table.

In [9]:
query = """
SELECT
    MIN(price)            AS min_price,
    ROUND(AVG(price), 2)  AS avg_price,
    MAX(price)            AS max_price
FROM order_items
"""

run(query)

,min_price,avg_price,max_price
0,0.85,120.65,6735.0


## 3. Delivered Orders
Count how many orders have been marked as `'delivered'`.

In [10]:
query = """
SELECT COUNT(*) AS delivered_orders
FROM orders
WHERE order_status = 'delivered'
"""

run(query)

,delivered_orders
0,96478


## 4. Low Review Ratings
Join to return `product_id`, `product_category_name`, and `review_score` for items that received a review score of `1`.


In [11]:
query = """
SELECT
    oi.product_id,
    p.product_category_name,
    r.review_score
FROM reviews AS r
JOIN order_items AS oi ON r.order_id = oi.order_id
JOIN products    AS p  ON oi.product_id = p.product_id
WHERE r.review_score = 1
"""

run(query)

,product_id,product_category_name,review_score
0,4e3550b4ebca5e32ccc3d32b3470e31f,cama_mesa_banho,1
1,c45d02cc82cd779835094de9b29272cb,cama_mesa_banho,1
2,c45d02cc82cd779835094de9b29272cb,cama_mesa_banho,1
3,c45d02cc82cd779835094de9b29272cb,cama_mesa_banho,1
4,55a1e3d39ace9c0b217f6c30b04d43c0,malas_acessorios,1
...,...,...,...
14230,d394d30215872be4cda2c67c1ce5b981,esporte_lazer,1
14231,0a57f7d2c983bcf8188589a5fea4a8da,relogios_presentes,1
14232,3150c2690393e7ea7c2b3acf2479ec8c,bebes,1
14233,d8342ac8b27a03e8d25280c296dfa830,moveis_decoracao,1


## 5. High-Value Categories
Find all product categories with an average purchase price above $150.


In [12]:
query = """
SELECT
    p.product_category_name,
    ROUND(AVG(oi.price), 2) AS avg_price
FROM order_items AS oi
JOIN products    AS p ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
HAVING AVG(oi.price) > 150
ORDER BY avg_price DESC
"""

run(query)

,product_category_name,avg_price
0,pcs,1098.34
1,portateis_casa_forno_e_cafe,624.29
2,eletrodomesticos_2,476.12
3,agro_industria_e_comercio,342.12
4,instrumentos_musicais,281.62
5,eletroportateis,280.78
6,portateis_cozinha_e_preparadores_de_alimentos,264.57
7,telefonia_fixa,225.69
8,construcao_ferramentas_seguranca,208.99
9,relogios_presentes,201.14


## 6. Average Freight
Find the average freight cost (`freight_value`) in `order_items`.

In [13]:
query = """
SELECT ROUND(AVG(freight_value), 2) AS avg_freight_value
FROM order_items
"""

run(query)

,avg_freight_value
0,19.99


## 7. Top 5 Prices
List the top 5 highest-priced items in `order_items` (`order_id` and `price`, highest to lowest).

In [14]:
query = """
SELECT order_id, price
FROM order_items
ORDER BY price DESC
LIMIT 5
"""

run(query)

,order_id,price
0,0812eb902a67711a1cb742b3cdaa65ae,6735.0
1,fefacc66af859508bf1a7934eab1e97f,6729.0
2,f5136e38d1a14a4dbd87dff67da82701,6499.0
3,a96610ab360d42a2e5335a3998b4718a,4799.0
4,199af31afc78c699f0dbf71fb178d4d4,4690.0


## 8. Specific Order Category Lookup
Join `order_items` and `products` to get the category name for `order_id = '00010242fe8c5a6d1ba2dd792cb16214'`.

In [15]:
query = """
SELECT
    oi.order_id,
    oi.product_id,
    p.product_category_name
FROM order_items AS oi
JOIN products    AS p ON oi.product_id = p.product_id
WHERE oi.order_id = '00010242fe8c5a6d1ba2dd792cb16214'
"""

run(query)

,order_id,product_id,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff


## 9. Review Score Distribution
Show the distribution of reviews (`review_score` and count) grouped by score, where `review_score` is `NOT NULL`.

In [16]:
query = """
SELECT
    review_score,
    COUNT(*) AS review_count
FROM reviews
WHERE review_score IS NOT NULL
GROUP BY review_score
ORDER BY review_score
"""

run(query)

,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


## 10. Category Max Price
Find the maximum price of a product sold in the `'cool_stuff'` category.

In [17]:
query = """
SELECT MAX(oi.price) AS max_cool_stuff_price
FROM order_items AS oi
JOIN products    AS p ON oi.product_id = p.product_id
WHERE p.product_category_name = 'cool_stuff'
"""

run(query)

,max_cool_stuff_price
0,3109.99


## Summary
- `DISTINCT`, `COUNT`, `MIN`/`AVG`/`MAX` give quick summaries of a single table.
- `JOIN` links tables through shared keys (`order_id`, `product_id`); `order_items` connects orders and reviews to products.
- `GROUP BY` + `HAVING` filters groups after aggregation (Q5), while `WHERE` filters individual rows before it.
- `ORDER BY ... LIMIT` returns the top-N rows (Q7).

In [18]:
conn.close()